In [ ]:
import os
import numpy as np
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import roc_utils as ru

In [ ]:
data_dir = '/Users/jk1/temp/bp_dci/kssg'
output_dir = '/Users/jk1/Downloads'

In [ ]:
bp_metrics_df = pd.DataFrame()
for dir in os.listdir(data_dir):
    if not dir.startswith('bp_timebin'):
        continue
    data_filenames = [f for f in os.listdir(os.path.join(data_dir, dir)) if f.endswith('.csv') and 'metrics' in f]
    for data_filename in data_filenames:
        data_is_normalized = int(('normalized' in data_filename) or ('normalised' in data_filename))
        df = pd.read_csv(os.path.join(data_dir, dir, data_filename))

        # if normalised_ in any column name, remove it
        df.columns = [col.replace('normalised_', '') for col in df.columns]

        df['normalized'] = data_is_normalized
        bp_metrics_df = pd.concat([bp_metrics_df, df], axis=0)
    
bp_metrics_df = bp_metrics_df.reset_index(drop=True)

In [ ]:
bp_metrics_df.shape

In [ ]:
bp_metrics_df.head()

In [ ]:
bp_metrics_df.columns

In [ ]:
metrics = ['negative_timebin', 'pNr', 'systole_median', 'diastole_median',
       'mitteldruck_median', 'systole_max', 'diastole_max', 'mitteldruck_max',
       'systole_min', 'diastole_min', 'mitteldruck_min', 'systole_cv',
       'diastole_cv', 'mitteldruck_cv', 'systole_arv', 'diastole_arv',
       'mitteldruck_arv', 'systole_ci', 'diastole_ci', 'mitteldruck_ci']

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

for tb_size in bp_metrics_df['timebin_size'].unique():
    for m in metrics:
        temp_df = bp_metrics_df[(bp_metrics_df['timebin_size'] == tb_size) & (bp_metrics_df['normalized'] == 0)][['label', m]]
        temp_df[m].replace([np.inf, -np.inf], np.nan, inplace=True)
        temp_df.dropna(subset=[m], inplace=True)
        if temp_df.label.nunique() == 1:
            continue
        score = roc_auc_score(temp_df['label'], temp_df[m])
        print(f'{tb_size}h, {m}: {score}')

In [ ]:
temp_df

In [ ]:
outcome = 'DCI_ischemia'
timebin_size = 4
data_is_normalized = 0
metric = 'diastole_median'

In [ ]:
all_colors_palette = sns.color_palette(['#f61067', '#049b9a', '#012D98', '#a76dfe', '#FFA987'], n_colors=5)
all_colors_palette

In [ ]:
from sklearn.metrics import roc_curve, auc, roc_auc_score
import matplotlib.patches as mpatches
from matplotlib.legend_handler import HandlerTuple

def plot_roc_curve(bp_metrics_df, timebin_size, metric, all_colors_palette,
    tick_label_size = 11,
    label_font_size = 13,
    n_samples = 10,
    plot_legend = True,
    fig=None):
    
    title = f"{metric}, {timebin_size}h timebin"
    
    normalized_metric_df = bp_metrics_df[(bp_metrics_df['timebin_size'] == timebin_size) & (bp_metrics_df['normalized'] == 1)]
    normalized_metric_df.dropna(subset=[metric], inplace=True)
    
    non_normalized_metric_df = bp_metrics_df[(bp_metrics_df['timebin_size'] == timebin_size) & (bp_metrics_df['normalized'] == 0)]
    non_normalized_metric_df.dropna(subset=[metric], inplace=True)

    norm_metric_fpr, norm_metric_tpr, norm_metric_thresholds = roc_curve(
        normalized_metric_df['label'],
        -1 * normalized_metric_df[metric],
        pos_label=1,
    )
    norm_metric_roc_auc = auc(norm_metric_fpr, norm_metric_tpr)
    
    non_norm_metric_fpr, non_norm_metric_tpr, non_norm_metric_thresholds = roc_curve(
        non_normalized_metric_df['label'],
        -1 * non_normalized_metric_df[metric],
        pos_label=1,
    )
    non_norm_metric_roc_auc = auc(non_norm_metric_fpr, non_norm_metric_tpr)
    
    if fig is None:
        fig, ax = plt.subplots(figsize=(10, 10))
    else:
        ax = fig.add_subplot(1, 1, 1)
    
    # Plot normalized metric 
    ru.plot_roc_bootstrap(X=-1 * normalized_metric_df[metric], y=normalized_metric_df['label'], ax=ax, 
                          pos_label=1,
                          n_bootstrap=n_samples,
                          random_state=42, show_ti=False)
    
    # set color 
    ax.get_lines()[0].set_color(all_colors_palette[0])
    ax.get_children()[0].set_facecolor(all_colors_palette[0])
    ax.get_children()[0].set_edgecolor(all_colors_palette[0])
    ax.get_children()[0].set_alpha(0.1)
    
    # Plot non-normalized metric
    ru.plot_roc_bootstrap(X=-1 * non_normalized_metric_df[metric], y=non_normalized_metric_df['label'], ax=ax, 
                          pos_label=1,
                          n_bootstrap=n_samples,
                          random_state=42, show_ti=False)
    
    # set color
    ax.get_lines()[2].set_color(all_colors_palette[3])
    ax.get_children()[3].set_facecolor(all_colors_palette[3])
    ax.get_children()[3].set_edgecolor(all_colors_palette[3])
    ax.get_children()[3].set_alpha(0.1)

    # Plot chance
    ax.plot([0, 1], [0, 1], color='grey', lw=1, linestyle='--', alpha=0.5)
    
    if plot_legend:
        legend_markers, _ = ax.get_legend_handles_labels()
        norm_label = f'Normalized (AUC = {norm_metric_roc_auc:.2f})'
        non_norm_label = f'Non-normalized (AUC = {non_norm_metric_roc_auc:.2f})'
        legend_labels = [norm_label, non_norm_label]

        sd1_patch = mpatches.Patch(color=all_colors_palette[0], alpha=0.3)
        sd2_patch = mpatches.Patch(color=all_colors_palette[3], alpha=0.3)
        sd_marker = (sd1_patch, sd2_patch)
        sd_labels = '95% CI'
        legend_markers.append(sd_marker)
        legend_labels.append(sd_labels)
        ax.legend(legend_markers, legend_labels, fontsize=label_font_size,
                  handler_map={tuple: HandlerTuple(ndivide=None)})
        
    else:
        # remove legend
        ax.get_legend().remove()
    
    
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('1 - Specificity (False Positive Rate)', fontsize=label_font_size)
    ax.set_ylabel('Sensitivity (True Positive Rate)', fontsize=label_font_size)
    ax.tick_params('x', labelsize=tick_label_size)
    ax.tick_params('y', labelsize=tick_label_size)
    
    plt.title(title)
    # remove suptitle
    plt.suptitle('')
    
    return ax, fig

In [ ]:
ax, fig = plot_roc_curve(bp_metrics_df, timebin_size, metric, all_colors_palette, plot_legend=True)

In [ ]:
# Save figure
# fig.savefig(os.path.join(output_dir, f'{outcome}_{timebin_size}h_{metric}_roc.png'), dpi=300, bbox_inches='tight')

In [ ]:
# create combined figure
sns.set_theme(style="whitegrid", context="paper", font_scale = 1)
cm = 1/2.54  # centimeters in inches
main_fig = plt.figure(figsize=(18 * cm, 10 * cm))

outcome = 'DCI_ischemia'
timebin_sizes = [8, 6]
metrics = ['CV_inter_eye_min_timebin_max', 'CV_inter_eye_min_timebin_min']
n_samples = 100

tick_label_size = 6
label_font_size = 7
subplot_number_font_size = 9
suptitle_font_size = 10
plot_subplot_titles = True
wspace = 0.15

subfigs = main_fig.subfigures(1, 2, width_ratios=[1, 1])
subfigs[0].subplots_adjust(wspace=wspace)
subfigs[1].subplots_adjust(wspace=wspace)

# CV_inter_eye_min_timebin_max at 8h
ax1, fig1 = plot_roc_curve(bp_metrics_df, outcome, timebin_sizes[0], metrics[0], all_colors_palette, plot_NPI=True, plot_legend=True, fig=subfigs[0],
                            tick_label_size=tick_label_size, label_font_size=label_font_size, n_samples=n_samples)
ax1.title.set_text('')
fig1.suptitle(f'A. CV (inter-eye min, max in timebin), {timebin_sizes[0]}h', fontsize=subplot_number_font_size, horizontalalignment='left', x=0, y=1.)

# CV_inter_eye_min_timebin_min at 6h
ax2, fig2 = plot_roc_curve(bp_metrics_df, outcome, timebin_sizes[1], metrics[1], all_colors_palette, plot_NPI=True, plot_legend=True, fig=subfigs[1],
                            tick_label_size=tick_label_size, label_font_size=label_font_size, n_samples=n_samples)
ax2.title.set_text('')
fig2.suptitle(f'B. CV (inter-eye min, min in timebin), {timebin_sizes[1]}h', fontsize=subplot_number_font_size, horizontalalignment='left', x=0, y=1.)

main_fig.tight_layout()

In [ ]:
# save figure
main_fig.savefig(os.path.join(output_dir, f'{outcome}_roc_combined.tiff'), dpi=1200, format='tiff', bbox_inches='tight')
# main_fig.savefig(os.path.join(output_dir, f'{outcome}_roc_combined.svg'), dpi=2400, format='svg', bbox_inches='tight')